# 91 — Reformulação multiclasse (smoke)

**Fase 1** da reformulação metodológica: responder à recomendação 0 da orientadora (treinar OneVsRest e multiclasse 7+other).

**Escopo (estritamente):**
- TF-IDF + LogReg multiclasse, 2 estratégias (`native` softmax e `ovr`).
- Smoke em 1k amostras de treino — val inteiro preserva distribuição natural.
- Reaproveita os splits binários existentes (`artifacts/splits/*.parquet`) — só muda o `y`.

**Fora do escopo:**
- BERT multiclasse, random search, cross-validation, Qwen, análise de FPs sistematizada, comparação com binário, teste no `test.parquet`.

## 0. Bootstrap (Colab + local)

Detecta o ambiente, monta o Drive em Colab, clona o repo e instala o pacote, e seleciona `SPLITS_DIR` / `RUNS_BASE` apropriados. Em Colab os splits sao extraidos de `My Drive/economy-classifier/colab_splits.zip` (gerado por `scripts/colab_pack.py`).

In [1]:
import subprocess
import sys
import zipfile
from pathlib import Path


def _run(cmd: list[str], description: str) -> None:
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr, file=sys.stderr)
        raise RuntimeError(f"{description} failed with exit code {result.returncode}")


IN_COLAB = "google.colab" in sys.modules
print("Ambiente:", "Google Colab" if IN_COLAB else "Local")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/almeidadm/economy-classifier.git"
    REPO_BRANCH = "main"
    DRIVE_FOLDER = "economy-classifier"

    DRIVE_BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = Path("/content/economy-classifier")

    if REPO_DIR.exists():
        _run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], "git fetch")
        _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], "git checkout")
        _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], "git reset")
    else:
        _run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], "git clone")

    _run(
        [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR),
         "--upgrade-strategy", "only-if-needed", "-q"],
        "pip install -e .",
    )
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))

    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    if not (SPLITS_DIR / "train.parquet").exists():
        zip_path = DRIVE_BASE / "colab_splits.zip"
        assert zip_path.exists(), (
            f"Falta colab_splits.zip em {DRIVE_BASE}. Rode scripts/colab_pack.py local e suba o zip."
        )
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(REPO_DIR / "artifacts")
        print("Splits extraidos em", SPLITS_DIR)

    RUNS_BASE = DRIVE_BASE / "runs"
else:
    REPO_DIR = Path.cwd().parent
    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    RUNS_BASE = REPO_DIR / "artifacts" / "runs"

RUNS_BASE.mkdir(parents=True, exist_ok=True)
print("SPLITS_DIR:", SPLITS_DIR)
print("RUNS_BASE :", RUNS_BASE)

Ambiente: Local
SPLITS_DIR: /home/diacrono/Documentos/repositorios/economy-classifier/artifacts/splits
RUNS_BASE : /home/diacrono/Documentos/repositorios/economy-classifier/artifacts/runs


## 1. Imports e configuracao

In [2]:
import pandas as pd

from economy_classifier.datasets import (
    MULTICLASS_TOP7,
    OTHER_LABEL,
    attach_multiclass_label,
)
from economy_classifier.evaluation import (
    compute_confusion_matrix,
    compute_multiclass_metrics,
)
from economy_classifier.tfidf import (
    TfidfMulticlassConfig,
    train_tfidf_multiclass,
)

In [3]:
SMOKE_N = 1000
SEED = 2026

# SPLITS_DIR e RUNS_BASE vem do bootstrap acima.
RUN_DIR_NATIVE = RUNS_BASE / "91_smoke_multiclasse_tfidf_native"
RUN_DIR_OVR = RUNS_BASE / "91_smoke_multiclasse_tfidf_ovr"
LABELS = list(MULTICLASS_TOP7) + [OTHER_LABEL]
print("labels:", LABELS)

labels: ['poder', 'colunas', 'mercado', 'esporte', 'mundo', 'cotidiano', 'ilustrada', 'outros']


In [4]:
train = pd.read_parquet(SPLITS_DIR / "train.parquet")
val = pd.read_parquet(SPLITS_DIR / "val.parquet")

assert "category" in train.columns, (
    "category column missing — re-run 01_preparacao_dados.ipynb to persist it."
)

train = attach_multiclass_label(train)
val = attach_multiclass_label(val)

print("train shape:", train.shape)
print("val shape:", val.shape)
print("\ntrain label_multi distribution:")
print(train["label_multi"].value_counts())
print("\nval label_multi distribution:")
print(val["label_multi"].value_counts())

train shape: (133030, 8)
val shape: (16629, 8)

train label_multi distribution:
label_multi
outros       25834
poder        17590
colunas      17273
mercado      16776
esporte      15708
mundo        13657
cotidiano    13597
ilustrada    12595
Name: count, dtype: int64[pyarrow]

val label_multi distribution:
label_multi
outros       3205
poder        2224
colunas      2184
mercado      2097
esporte      1971
mundo        1742
cotidiano    1713
ilustrada    1493
Name: count, dtype: int64[pyarrow]


In [5]:
# Stratified smoke sample on train. Val is kept whole to preserve natural
# distribution (CLAUDE.md guard-rail: no balancing on val/test).
per_class = max(1, SMOKE_N // len(LABELS))
smoke_parts = []
for label in LABELS:
    pool = train[train["label_multi"] == label]
    take = min(per_class, len(pool))
    smoke_parts.append(pool.sample(n=take, random_state=SEED))
train_smoke = pd.concat(smoke_parts).sample(frac=1, random_state=SEED)
print("smoke train shape:", train_smoke.shape)
print(train_smoke["label_multi"].value_counts())

smoke train shape: (1000, 8)
label_multi
mundo        125
cotidiano    125
esporte      125
ilustrada    125
colunas      125
poder        125
outros       125
mercado      125
Name: count, dtype: int64[pyarrow]


In [6]:
config_native = TfidfMulticlassConfig(strategy="native")
result_native = train_tfidf_multiclass(
    train_smoke, val,
    label_column="label_multi",
    run_dir=RUN_DIR_NATIVE,
    config=config_native,
)
print("metrics:", result_native["metrics"])
print("vocab size:", result_native["vocabulary_size"])
print("timing:", result_native["timing"])

metrics: {'accuracy': 0.7171, 'macro_f1': 0.7197}
vocab size: 50000
timing: {'train_seconds': 3.91, 'inference_seconds': 14.34}


In [7]:
preds_native = result_native["predictions"]
metrics_native = compute_multiclass_metrics(
    preds_native["y_true"], preds_native["y_pred"], labels=LABELS,
)
cm_native = compute_confusion_matrix(
    preds_native["y_true"], preds_native["y_pred"],
    labels=LABELS, normalize="true",
)
print("NATIVE — macro_f1:", metrics_native["macro_f1"],
      "weighted_f1:", metrics_native["weighted_f1"],
      "accuracy:", metrics_native["accuracy"])
print("\nper-class F1:")
for k, v in metrics_native["per_class_f1"].items():
    print(f"  {k:12s} {v}")
print("\nconfusion matrix (rows=true, cols=pred, normalized by row):")
cm_native.round(3)

NATIVE — macro_f1: 0.7197 weighted_f1: 0.7083 accuracy: 0.7171

per-class F1:
  poder        0.7681
  colunas      0.5244
  mercado      0.7122
  esporte      0.8973
  mundo        0.808
  cotidiano    0.7217
  ilustrada    0.7182
  outros       0.6074

confusion matrix (rows=true, cols=pred, normalized by row):


,poder,colunas,mercado,esporte,mundo,cotidiano,ilustrada,outros
poder,0.834,0.030,0.042,0.003,0.016,0.051,0.006,0.018
colunas,0.144,0.448,0.121,0.051,0.053,0.030,0.097,0.056
mercado,0.080,0.044,0.753,0.001,0.018,0.032,0.016,0.056
esporte,0.020,0.010,0.012,0.895,0.018,0.028,0.009,0.009
mundo,0.016,0.022,0.031,0.002,0.866,0.029,0.021,0.011
cotidiano,0.037,0.033,0.030,0.003,0.018,0.786,0.037,0.055
ilustrada,0.016,0.019,0.011,0.005,0.028,0.027,0.869,0.026
outros,0.036,0.083,0.080,0.019,0.059,0.087,0.139,0.498


In [8]:
config_ovr = TfidfMulticlassConfig(strategy="ovr")
result_ovr = train_tfidf_multiclass(
    train_smoke, val,
    label_column="label_multi",
    run_dir=RUN_DIR_OVR,
    config=config_ovr,
)
print("metrics:", result_ovr["metrics"])
print("vocab size:", result_ovr["vocabulary_size"])
print("timing:", result_ovr["timing"])

metrics: {'accuracy': 0.7123, 'macro_f1': 0.7136}
vocab size: 50000
timing: {'train_seconds': 13.51, 'inference_seconds': 15.72}


In [9]:
preds_ovr = result_ovr["predictions"]
metrics_ovr = compute_multiclass_metrics(
    preds_ovr["y_true"], preds_ovr["y_pred"], labels=LABELS,
)
cm_ovr = compute_confusion_matrix(
    preds_ovr["y_true"], preds_ovr["y_pred"],
    labels=LABELS, normalize="true",
)
print("OVR — macro_f1:", metrics_ovr["macro_f1"],
      "weighted_f1:", metrics_ovr["weighted_f1"],
      "accuracy:", metrics_ovr["accuracy"])
print("\nper-class F1:")
for k, v in metrics_ovr["per_class_f1"].items():
    print(f"  {k:12s} {v}")
print("\nconfusion matrix (rows=true, cols=pred, normalized by row):")
cm_ovr.round(3)

OVR — macro_f1: 0.7136 weighted_f1: 0.7015 accuracy: 0.7123

per-class F1:
  poder        0.7636
  colunas      0.5046
  mercado      0.707
  esporte      0.8971
  mundo        0.8069
  cotidiano    0.7199
  ilustrada    0.7137
  outros       0.5961

confusion matrix (rows=true, cols=pred, normalized by row):


,poder,colunas,mercado,esporte,mundo,cotidiano,ilustrada,outros
poder,0.841,0.025,0.041,0.004,0.015,0.051,0.007,0.017
colunas,0.157,0.417,0.131,0.053,0.054,0.033,0.102,0.053
mercado,0.086,0.039,0.758,0.001,0.019,0.031,0.017,0.049
esporte,0.020,0.009,0.011,0.901,0.018,0.026,0.008,0.007
mundo,0.017,0.018,0.033,0.002,0.867,0.030,0.022,0.010
cotidiano,0.039,0.030,0.034,0.003,0.018,0.789,0.040,0.047
ilustrada,0.017,0.015,0.011,0.005,0.029,0.026,0.875,0.022
outros,0.037,0.079,0.087,0.021,0.060,0.093,0.145,0.478


## Conclusao da Fase 1 (smoke)

A pipeline multiclasse 7+other roda end-to-end: dados → splits existentes → mapeamento → treino → avaliacao → persistencia. Nada alem disso foi feito.

**O que esta entrega habilita (insumos para decidir Fase 2):**
- Macro-F1 das 8 classes em val, com 1k de treino balanceado.
- Matriz de confusao normalizada — base para a analise sistematizada de FPs (recomendacao 3 da orientadora).
- Comparacao native vs OvR para decidir a estrategia padrao na Fase 2 (BERT multiclasse).

**O que foi explicitamente deixado de fora:**
- BERT multiclasse, random search, cross-validation 90/10, Qwen, ensembles multiclasse, teste no `test.parquet`, atualizacao do `artigo_ieee.md`.
- Comparacao com binario — vira contraste artificial enquanto nao houver BERT e CV.

**Criterio de parada da Fase 1 atingido:** humano (orientadora) le este notebook e decide o escopo da Fase 2.

In [10]:
import numpy as np

def top_offdiag(cm: pd.DataFrame, k: int = 3) -> list[tuple[str, str, float]]:
    matrix = cm.to_numpy().copy()
    np.fill_diagonal(matrix, 0.0)
    flat = [
        (cm.index[i], cm.columns[j], matrix[i, j])
        for i in range(matrix.shape[0]) for j in range(matrix.shape[1])
    ]
    return sorted(flat, key=lambda x: x[2], reverse=True)[:k]


print("=== Resumo Fase 1 — TF-IDF + LogReg multiclasse (smoke 1k) ===\n")
print(f"{'estrategia':<12} {'macro_f1':<10} {'weighted_f1':<12} {'accuracy':<10}")
print(f"{'native':<12} {metrics_native['macro_f1']:<10} "
      f"{metrics_native['weighted_f1']:<12} {metrics_native['accuracy']:<10}")
print(f"{'ovr':<12} {metrics_ovr['macro_f1']:<10} "
      f"{metrics_ovr['weighted_f1']:<12} {metrics_ovr['accuracy']:<10}")

print("\n--- top-3 confusoes off-diagonal (NATIVE) ---")
for true, pred, score in top_offdiag(cm_native, k=3):
    print(f"  {true:10s} -> {pred:10s} : {score:.3f}")

print("\n--- top-3 confusoes off-diagonal (OVR) ---")
for true, pred, score in top_offdiag(cm_ovr, k=3):
    print(f"  {true:10s} -> {pred:10s} : {score:.3f}")

=== Resumo Fase 1 — TF-IDF + LogReg multiclasse (smoke 1k) ===

estrategia   macro_f1   weighted_f1  accuracy  
native       0.7197     0.7083       0.7171    
ovr          0.7136     0.7015       0.7123    

--- top-3 confusoes off-diagonal (NATIVE) ---
  colunas    -> poder      : 0.144
  outros     -> ilustrada  : 0.139
  colunas    -> mercado    : 0.121

--- top-3 confusoes off-diagonal (OVR) ---
  colunas    -> poder      : 0.157
  outros     -> ilustrada  : 0.145
  colunas    -> mercado    : 0.131
